# Cashflow is genuinely healthy or propped up by debt?

In [1]:
from pathlib import Path

project_root = Path.cwd().parent
data_dir = project_root / "data" / "raw"

In [2]:
import sys
sys.path.append('..')
from src.bookstore_health_analysis.data_loader import load_all_data

dfs = load_all_data(data_dir)

In [3]:
chk = dfs['bookstore_checking_balanced_dataset']
netflow = chk['deposit'] - chk['withdrawal']

In [4]:
print((netflow - chk['net_change']).abs().max())

0.0


In [5]:
deposits_by_desc = (
    chk.groupby('description')['deposit']
    .sum()
    .sort_values(ascending=False)
)

withdrawals_by_desc = (
    chk.groupby('description')['withdrawal']
    .sum()
    .sort_values(ascending=False)
)

print(deposits_by_desc.head(15))
print(withdrawals_by_desc.head(15))

description
Sales sweep deposit                 3185888.28
Refunds / chargeback reversal         10628.38
Inventory supplier payment                0.00
Misc operating expense                    0.00
Payroll                                   0.00
Store & warehouse rent                    0.00
Utilities (power/water/internet)          0.00
Name: deposit, dtype: float64
description
Payroll                             1963694.91
Store & warehouse rent              1636140.00
Inventory supplier payment          1398039.80
Utilities (power/water/internet)     122198.46
Misc operating expense                86596.88
Refunds / chargeback reversal             0.00
Sales sweep deposit                       0.00
Name: withdrawal, dtype: float64


In [6]:
chk = dfs['bookstore_checking_balanced_dataset'].sort_values('date')

opening = chk['rolling_balance'].iloc[0] - chk['net_change'].iloc[0]
print("opening balance:", opening)
print("closing balance:", chk['rolling_balance'].iloc[-1])
print("lowest balance:", chk['rolling_balance'].min())

balance_q = chk.groupby(chk['date'].dt.to_period('Q'))['rolling_balance'].last()
print(balance_q)

opening balance: 75000.0
closing balance: -1935153.39
lowest balance: -1943238.98
date
2019Q1       3320.41
2019Q2     -60327.80
2019Q3    -138528.22
2019Q4    -215154.20
2020Q1    -323419.60
2020Q2    -400975.32
2020Q3    -492510.90
2020Q4    -581983.55
2021Q1    -657675.87
2021Q2    -719016.50
2021Q3    -817099.71
2021Q4    -885778.42
2022Q1    -973132.94
2022Q2   -1073279.68
2022Q3   -1133700.78
2022Q4   -1212447.89
2023Q1   -1304488.71
2023Q2   -1400795.86
2023Q3   -1478583.99
2023Q4   -1581781.78
2024Q1   -1668838.46
2024Q2   -1766527.34
2024Q3   -1847245.80
2024Q4   -1935153.39
Freq: Q-DEC, Name: rolling_balance, dtype: float64


In [7]:
sales = dfs['bookstore_sales']
chk = dfs['bookstore_checking_balanced_dataset']

sweeps = chk[chk['description'] == 'Sales sweep deposit']
sweeps_q = sweeps.groupby(sweeps['date'].dt.to_period('Q'))['deposit'].sum()

sales_q = sales.groupby(sales['date'].dt.to_period('Q'))[['revenue', 'net_revenue']].sum()

compare = sales_q.join(sweeps_q.rename('sweeps'))
compare['sweep_to_gross'] = compare['sweeps'] / compare['revenue']
compare['sweep_to_net'] = compare['sweeps'] / compare['net_revenue']
compare

,revenue,net_revenue,sweeps,sweep_to_gross,sweep_to_net
date,,,,,
2019Q1,516338.28,490521.35,126060.36,0.244143,0.256993
2019Q2,615524.98,584748.73,128143.23,0.208185,0.219142
2019Q3,616114.33,585308.58,142481.01,0.231257,0.243429
2019Q4,742654.91,705522.04,135841.49,0.182913,0.192540
2020Q1,365486.37,347212.06,93904.29,0.256930,0.270452
2020Q2,421398.42,400328.41,94688.97,0.224702,0.236528
2020Q3,431929.89,410333.44,96128.59,0.222556,0.234269
2020Q4,512393.58,486774.00,111654.40,0.217907,0.229376
2021Q1,441220.96,419159.92,110974.65,0.251517,0.264755


In [8]:
pay = dfs['bookstore_payroll']
chk = dfs['bookstore_checking_balanced_dataset']
chk_payroll = chk[chk['description'] == 'Payroll']
chk_total = chk_payroll['withdrawal'].sum()

print("checking payroll total:", round(chk_total))
print("payroll rows in checking:", len(chk_payroll),
      "| pay periods in payroll file:", pay['pay_period_start'].nunique())

pay['gross_plus_benefits'] = pay['gross_pay'] + pay['employee_benefits']
for col in ['gross_pay', 'net_pay', 'gross_plus_benefits']:
    print(col, round(pay[col].sum()), "ratio to checking:", round(pay[col].sum() / chk_total, 3))

# quarter by quarter
chk_q = chk_payroll.groupby(chk_payroll['date'].dt.to_period('Q'))['withdrawal'].sum().rename('checking')
pay_q = pay.groupby(pay['pay_period_start'].dt.to_period('Q'))[['gross_pay', 'net_pay', 'gross_plus_benefits']].sum()
rec = pay_q.join(chk_q)
for col in ['gross_pay', 'net_pay', 'gross_plus_benefits']:
    rec[col + '_ratio'] = rec['checking'] / rec[col]
rec

checking payroll total: 1963695
payroll rows in checking: 157 | pay periods in payroll file: 157
gross_pay 6770082 ratio to checking: 3.448
net_pay 5411523 ratio to checking: 2.756
gross_plus_benefits 6855866 ratio to checking: 3.491


,gross_pay,net_pay,gross_plus_benefits,checking,gross_pay_ratio,net_pay_ratio,gross_plus_benefits_ratio
pay_period_start,,,,,,,
2019Q1,132983.28,107064.83,133900.90,86397.35,0.649686,0.806963,0.645234
2019Q2,151164.56,121733.13,152177.09,73630.00,0.487085,0.604848,0.483844
2019Q3,188608.22,152103.33,189654.77,87491.32,0.463879,0.575210,0.461319
2019Q4,186611.73,150221.64,187918.79,69956.81,0.374879,0.465691,0.372272
2020Q1,146407.20,117466.78,147823.09,73636.34,0.502956,0.626869,0.498138
2020Q2,130261.84,104460.42,131574.00,65753.00,0.504776,0.629454,0.499742
2020Q3,191514.05,153790.79,193232.84,78237.19,0.408519,0.508725,0.404886
2020Q4,178552.43,143220.88,180316.23,74786.84,0.418851,0.522178,0.414754
2021Q1,223827.92,178976.53,226599.55,64959.47,0.290221,0.362950,0.286671


In [9]:
loans = dfs['bookstore_loans']
print(loans['loan_name'].value_counts())
print(loans.groupby('quarter_end').size().value_counts())

loan_q = loans.groupby(loans['quarter_end'].dt.to_period('Q')).agg(
    principal=('principal_outstanding', 'sum'),
    repayment=('repayment', 'sum'),
    interest=('interest_accrued', 'sum'),
)
loan_q

loan_name
Business Line of Credit    24
Government Relief Loan     20
Name: count, dtype: int64
2    20
1     4
Name: count, dtype: int64


,principal,repayment,interest
quarter_end,,,
2019Q1,10715.37,17187.85,187.52
2019Q2,2326.04,17245.21,40.71
2019Q3,0.00,19206.88,0.00
2019Q4,0.00,21753.93,0.00
2020Q1,55393.43,13342.21,269.39
2020Q2,56722.70,16802.95,292.65
2020Q3,45512.96,23546.66,96.48
2020Q4,54702.55,8336.41,257.29
2021Q1,70099.62,9383.03,526.74


## Findings: Is Cash Flow Healthy, or Propped Up by Debt?

The checking account lost money every quarter, going from a 75K opening balance
to about -1.94M, an average loss of roughly 84K a quarter. Deposits covered
about 61% of withdrawals. There is no visible COVID effect on the burn, which
looks about the same in 2019 and 2024.

That does not mean the business lost $2M. The account sees only about 23% of
sales (sales sweeps) and pays only about 29% of the payroll file's totals. The
checking data seems to be on its own scale and cannot be reconciled with the
rest of the dataset, which describes itself as a bookkeeping simulation. I treat
it as a partial view, not as proof that the business is unprofitable.

Debt tells a separate story. Loans were near zero in 2019, jumped to about 55K
in 2020Q1 (most likely a 40K relief loan plus a line of credit balance), and
never came back down, peaking at about 132K in 2024Q3. Borrowing continued
after revenue recovered, and interest grew to about 2K a quarter.

Debt is not what keeps the account running. Peak debt is about 15 times
smaller than the account's deficit, and no loan proceeds appear in the checking
deposits.

**Bottom line**: cash flow cannot be called healthy from this account, but it
is not propped up by loans either. What is worth flagging to the owner is that
debt taken on in 2020 was never paid down.

**Limitations**: checking could not be reconciled with sales or payroll, and
the inventory file was not used.